In [1]:
!pip install sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
import json
import re

def benzerlik_analizi_yap():
    """
    1. Kullanıcıdan JSON girdi dosyası yüklemesini ister.
    2. Her kayıt için:
       - Soru  <-> Öğrenci Cevabı   kosinüs benzerliği
       - Doğru Cevap <-> Öğrenci Cevabı kosinüs benzerliği
    3. Sonuçları .txt dosyasına kaydedip indirir.
    """

    # ── 1. Dosya yükleme ──────────────────────────────────────────────────────
    print("Lütfen JSON girdi dosyasını yükleyin...")
    yuklenen = files.upload()

    if not yuklenen:
        print("Hiçbir dosya yüklenmedi. İşlem iptal edildi.")
        return

    dosya_adi = list(yuklenen.keys())[0]
    icerik    = yuklenen[dosya_adi].decode("utf-8")

    # ── 2. JSON ayrıştırma ────────────────────────────────────────────────────
    try:
        veriler = json.loads(icerik)
        # Tek dict geldiyse listeye çevir
        if isinstance(veriler, dict):
            veriler = [veriler]
    except json.JSONDecodeError as e:
        print(f"JSON ayrıştırma hatası: {e}")
        return

    # ── 3. Model yükleme ──────────────────────────────────────────────────────
    print("\nModel yükleniyor: emrecan/bert-base-turkish-cased-mean-nli-stsb-tr")
    model = SentenceTransformer('emrecan/bert-base-turkish-cased-mean-nli-stsb-tr')
    print("Model başarıyla yüklendi.\n")

    # ── 4. Benzerlik hesaplama ────────────────────────────────────────────────
    satirlar = []

    for idx, kayit in enumerate(veriler, start=1):
        girdi = kayit.get("input", "")

        # input alanından Soru / Doğru Cevap / Öğrenci Cevabı'nı ayıkla
        soru_eslesme          = re.search(r"Soru:\s*(.*?)(?=\nDoğru Cevap:)", girdi, re.DOTALL)
        dogru_cevap_eslesme   = re.search(r"Doğru Cevap:\s*(.*?)(?=\nÖğrenci Cevabı:)", girdi, re.DOTALL)
        ogrenci_cevap_eslesme = re.search(r"Öğrenci Cevabı:\s*(.*)", girdi, re.DOTALL)

        if not (soru_eslesme and dogru_cevap_eslesme and ogrenci_cevap_eslesme):
            print(f"[{idx}] Kayıtta beklenen alanlar bulunamadı, atlanıyor.")
            continue

        soru          = soru_eslesme.group(1).strip()
        dogru_cevap   = dogru_cevap_eslesme.group(1).strip()
        ogrenci_cevap = ogrenci_cevap_eslesme.group(1).strip()

        # Embedding'leri hesapla
        embeddings = model.encode([soru, dogru_cevap, ogrenci_cevap])
        emb_soru          = embeddings[0].reshape(1, -1)
        emb_dogru_cevap   = embeddings[1].reshape(1, -1)
        emb_ogrenci_cevap = embeddings[2].reshape(1, -1)

        cos_soru_ogrenci  = cosine_similarity(emb_soru, emb_ogrenci_cevap)[0][0]
        cos_dogru_ogrenci = cosine_similarity(emb_dogru_cevap, emb_ogrenci_cevap)[0][0]

        print(f"[{idx}] Soru ↔ Öğrenci Cevabı  : {cos_soru_ogrenci:.4f}")
        print(f"[{idx}] Doğru Cevap ↔ Öğrenci  : {cos_dogru_ogrenci:.4f}\n")

        # Satır oluştur
        satirlar.append(
            f'{{\n'
            f'  "instruction": "Öğrenci cevabını analiz et, SBERT benzerlik skorlarını değerlendir ve 0.0-1.0 arasında bir puan ver.",\n'
            f'  "input": "Soru: {soru}\\n'
            f'Doğru Cevap: {dogru_cevap}\\n'
            f'Öğrenci Cevabı: {ogrenci_cevap}\\n'
            f'SBERT Skorları: Soru-Öğrenci Cevabı Benzerliği: {cos_soru_ogrenci:.4f}, '
            f'Öğrenci Cevabı-Doğru Cevap benzerliği: {cos_dogru_ogrenci:.4f}",\n'
            f'  "output": " ANALİZ VE PUAN "\n'
            f'}}' # <--- Buradaki virgülü sildik
        )

    # ── 5. TXT'e kaydet ve indir ──────────────────────────────────────────────
    # Yazdırma aşaması
    cikti_dosyasi = "benzerlik_sonuclari.json"
    with open(cikti_dosyasi, "w", encoding="utf-8") as f:
        f.write("[\n") # Tek parantez genelde yeterlidir ama [[ istediyseniz öyle kalsın
        f.write(",\n".join(satirlar)) # Elemanların arasına virgül koyar, en sona koymaz
        f.write("\n]")

    print(f"Sonuçlar '{cikti_dosyasi}' dosyasına kaydedildi. İndiriliyor...")
    files.download(cikti_dosyasi)


In [3]:
# ── Fonksiyonu çalıştır ───────────────────────────────────────────────────────
benzerlik_analizi_yap()

Lütfen JSON girdi dosyasını yükleyin...


Saving Dataset1.json to Dataset1.json

Model yükleniyor: emrecan/bert-base-turkish-cased-mean-nli-stsb-tr


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model başarıyla yüklendi.

[1] Soru ↔ Öğrenci Cevabı  : 0.6945
[1] Doğru Cevap ↔ Öğrenci  : 0.9295

[2] Soru ↔ Öğrenci Cevabı  : 0.7050
[2] Doğru Cevap ↔ Öğrenci  : 0.8794

[3] Soru ↔ Öğrenci Cevabı  : 0.7434
[3] Doğru Cevap ↔ Öğrenci  : 0.9104

[4] Soru ↔ Öğrenci Cevabı  : 0.6551
[4] Doğru Cevap ↔ Öğrenci  : 0.8538

[5] Soru ↔ Öğrenci Cevabı  : 0.6535
[5] Doğru Cevap ↔ Öğrenci  : 0.9072

[6] Soru ↔ Öğrenci Cevabı  : 0.6990
[6] Doğru Cevap ↔ Öğrenci  : 0.7276

[7] Soru ↔ Öğrenci Cevabı  : 0.6945
[7] Doğru Cevap ↔ Öğrenci  : 0.8196

[8] Soru ↔ Öğrenci Cevabı  : 0.7785
[8] Doğru Cevap ↔ Öğrenci  : 0.7428

[9] Soru ↔ Öğrenci Cevabı  : 0.7440
[9] Doğru Cevap ↔ Öğrenci  : 0.7360

[10] Soru ↔ Öğrenci Cevabı  : 0.6609
[10] Doğru Cevap ↔ Öğrenci  : 0.7741

[11] Soru ↔ Öğrenci Cevabı  : 0.7572
[11] Doğru Cevap ↔ Öğrenci  : 0.6758

[12] Soru ↔ Öğrenci Cevabı  : 0.6573
[12] Doğru Cevap ↔ Öğrenci  : 0.8681

[13] Soru ↔ Öğrenci Cevabı  : 0.7301
[13] Doğru Cevap ↔ Öğrenci  : 0.8605

[14] Soru ↔ Öğre

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>